# Axolotl Telencephalon: raw data to spatial domains

This Stereo-seq example runs the reviewed SpaDOT recipe from raw counts through
preprocessing, 99 training updates, KMeans domains, and temporal transport
analysis. The raw file is `AxolotlTelencephalon_data/Axolotl.h5ad` beside this notebook.
The feature quota is **500 / 500 / 1000 genes per timepoint**,
with fixed final cluster counts **[6, 7, 8]**.

`--feature_panel fixed99_panels/AxolotlTelencephalon` performs the exact historical
panel replay in Python. It checks the raw file and archived SPARK-X candidate
rank hashes, recomputes the 12 KMeans pseudoregion gene rankings, and requires
an exact match to the saved gene lists. The archived candidate ranks were
originally produced by R SPARK-X; **no R runtime is used here**. Omitting
`--feature_panel` creates a new Python SPARK-X panel that may differ.

Run the notebook from the repository root or `examples/`. Set
`SPADOT_RUN_NAME` to a new name for each run and `SPADOT_DEVICE` to an
available GPU (default `cuda:0`). Start Jupyter with
`PYTHONHASHSEED=2026`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, and numerical
thread limits of 1. Generated files go under
`examples/fixed99_runs/<run name>/AxolotlTelencephalon/`; earlier outputs are preserved.


In [ ]:
import json
import os
import re
import sys
from argparse import Namespace
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

examples = Path.cwd().resolve()
if not (examples / "fixed99_panels").is_dir():
    examples = examples / "examples"
if not (examples / "fixed99_panels").is_dir():
    raise FileNotFoundError("Run from the repository root or examples/")
sys.path.insert(0, str(examples.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import yaml
from threadpoolctl import threadpool_limits
import SpaDOT

dataset = "AxolotlTelencephalon"
raw_file = examples / "AxolotlTelencephalon_data" / "Axolotl.h5ad"
panel_dir = examples / "fixed99_panels" / dataset
config_file = examples.parent / "SpaDOT" / "config.yaml"
run_name = os.environ.get("SPADOT_RUN_NAME", "python_replay")
if not re.fullmatch(r"[A-Za-z0-9_-]+", run_name):
    raise ValueError("SPADOT_RUN_NAME must contain only letters, numbers, - or _")
output_dir = examples / "fixed99_runs" / run_name / dataset
output_dir.mkdir(parents=True, exist_ok=True)
device = os.environ.get("SPADOT_DEVICE", "cuda:0")

if not raw_file.is_file():
    raise FileNotFoundError(raw_file)
panel = json.loads((panel_dir / "panel.json").read_text())
config = yaml.safe_load(config_file.read_text())
assert config["epoch_start"] == 1
assert config["epoch_stop_exclusive"] == 100
assert config["kl_cycle_epochs"] == 30
assert config["seed"] == 2026
torch.use_deterministic_algorithms(True, warn_only=False)
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if device.startswith("cuda"):
    assert torch.cuda.is_available()
    torch.cuda.set_device(int(device.split(":")[1]))
print("SpaDOT source:", SpaDOT.__file__)
print("Raw:", raw_file)
print("Output:", output_dir)
print("Device:", device)


## 1. Preprocess raw counts

The panel selector checks the archived candidate-rank snapshots and
recomputes each timepoint's pseudoregion list in Python. It forms the sorted
union from **raw counts**, then independently applies
`normalize_total(target_sum=1e-4)`, `log1p`, and `scale` at each
timepoint. Raw selected counts remain in `layers["counts"]`.


In [ ]:
input_file = output_dir / f"preprocessed_fixed99_{raw_file.name}"
if input_file.exists():
    raise FileExistsError(f"Input already exists: {input_file}. Choose a new SPADOT_RUN_NAME.")
preprocess_args = Namespace(
    data=str(raw_file), output_dir=str(output_dir),
    prefix="preprocessed_fixed99_", feature_selection=True,
    n_features="500,500,1000", feature_panel=str(panel_dir),
)
with threadpool_limits(limits=1):
    SpaDOT.preprocess(preprocess_args)
selection = json.loads((output_dir / "SVG_selection.json").read_text())
assert selection["method"] == "python_replay_of_historical_ranks"
assert selection["n_features"] == panel["selected_genes_per_timepoint"]
prepared = ad.read_h5ad(input_file, backed="r")
assert list(prepared.shape) == panel["reference_input_shape"]
assert prepared.var_names.tolist() == (panel_dir / "union_genes.txt").read_text().splitlines()
print("Prepared:", input_file, prepared.shape)
prepared.file.close()


## 2. Train SpaDOT

The checked-in configuration executes epochs 1 through 99 with a 30-epoch
cyclic SVGP KL weight, seed 2026, and the Python/NumPy OT backend. The latent
file and model are from the **executed epoch 99**. A fresh run directory
prevents overwriting a previous attempt.


In [ ]:
latent_file = output_dir / "latent.h5ad"
if latent_file.exists():
    raise FileExistsError(f"Latent already exists: {latent_file}. Choose a new SPADOT_RUN_NAME.")
train_args = Namespace(
    data=str(input_file), output_dir=str(output_dir),
    prefix="", config=str(config_file), save_model=True, device=device,
)
with threadpool_limits(limits=1):
    SpaDOT.train(train_args)
loss = pd.read_csv(output_dir / "loss.csv", index_col=0)
assert [int(epoch) for epoch in loss.index] == list(range(1, 100))
latent = ad.read_h5ad(latent_file, backed="r")
assert latent.shape == (panel["reference_input_shape"][0], config["z_dim"])
print("Executed epochs:", len(loss), "Latent:", latent.shape)
latent.file.close()


## 3. Analyze spatial domains and transitions

The evaluated readout uses KMeans++/Lloyd, seed 2026, `n_init=10`, and
[6, 7, 8] clusters across the timepoints. `SpaDOT.analyze` writes one
spatial-domain image per timepoint and transition dotplots.


In [ ]:
timepoints = ['t0', 't1', 't2']
clusters = [6, 7, 8]
analyze_args = Namespace(
    data=str(latent_file), output_dir=str(output_dir), prefix="",
    n_clusters=clusters, seed=2026, n_init=10,
)
with threadpool_limits(limits=1):
    SpaDOT.analyze(analyze_args)
for tp in timepoints:
    assert (output_dir / f"{tp}_domains.png").is_file()
print("Spatial plots:", [str(output_dir / f"{tp}_domains.png") for tp in timepoints])
print("Transition plots:", sorted(str(path) for path in output_dir.glob("transition_dotplot_*.png")))


In [ ]:
from IPython.display import Image, display

for tp in timepoints:
    display(Image(filename=str(output_dir / f"{tp}_domains.png")))
